In [1]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_path = './cifake/train'
train_dataset = ImageFolder(root=train_path, transform=transform)

In [6]:
print(f'Number of training samples: {len(train_dataset)}')
print(f'Number of classes: {len(train_dataset.classes)}')
print(f'Class names: {train_dataset.classes}')
print(f"Mapping: {train_dataset.class_to_idx}")

Number of training samples: 100000
Number of classes: 2
Class names: ['FAKE', 'REAL']
Mapping: {'FAKE': 0, 'REAL': 1}


In [2]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [14]:
imagens, rotulos = next(iter(train_loader))

print(f'Batch de imagens: {imagens.shape}')
print(f'Batch de rótulos: {rotulos.shape}')
print(f'Rótulos únicos no batch: {rotulos}')    

Batch de imagens: torch.Size([64, 3, 32, 32])
Batch de rótulos: torch.Size([64])
Rótulos únicos no batch: tensor([0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1,
        1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0])


In [3]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(16,32,3,padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(32 * 16 * 16, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x
    
model = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0 
    total = 0

    print(f'Epoch {epoch+1}/{epochs}')

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f'Loss: {running_loss/len(train_loader):.4f}, Accuracy: {correct/total:.4f}')

Epoch 1/10
Loss: 0.2967, Accuracy: 0.8730
Epoch 2/10
Loss: 0.2055, Accuracy: 0.9204
Epoch 3/10


KeyboardInterrupt: 

In [9]:
torch.save(model.state_dict(), 'simple_cnn_cifake_model.pth')

In [ ]:
model = SimpleCNN().to(device)
model.load_state_dict(torch.load('simple_cnn_cifake_model.pth', map_location=device))

model.eval()

test_path = './cifake/test'
test_dataset = ImageFolder(root=test_path, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

all_preds = []
all_labels = []


with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        output = model(inputs)
        _, predicted = torch.max(output, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
print(f"Accurarcy: {acc * 100:.2f}%\n")

print("Report by Class:")
print(classification_report(all_labels, all_preds, target_names=test_dataset.classes))

--------------------------------------------------------------------------------
Accurarcy: 94.83%

Report by Class:
              precision    recall  f1-score   support

        FAKE       0.96      0.93      0.95     10000
        REAL       0.94      0.96      0.95     10000

    accuracy                           0.95     20000
   macro avg       0.95      0.95      0.95     20000
weighted avg       0.95      0.95      0.95     20000

